In [ ]:
import fitz
import os
from dotenv import load_dotenv
from groq import Groq
import re
import random
from tqdm.auto import tqdm
import numpy as np
import torch
import pandas as pd
from time import perf_counter as timer
from langchain_text_splitters import HTMLSemanticPreservingSplitter

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
load_dotenv()
api_key = os.getenv("keys")
print(api_key)

In [ ]:
gpt4_questions = [
    "When was the University of Sharjah established, and under whose leadership?",
    "What is the vision and mission of the University of Sharjah?",
    "How many colleges does the University of Sharjah have, and how many academic programs does it offer?"
    "What are the admission requirements for undergraduate and graduate programs?",
    "What is the grading system and how is the GPA calculated at the University?",
    "What student support services are available, such as academic advising, counseling, and career services?",
    "What financial assistance options are available for students, and what are the methods for paying fees?",
    "What is the role of the Deanship of Student Affairs, and what services does it provide?",
    "What sports and extracurricular activities are available for students?",
    "What are the rules and consequences for academic dishonesty (cheating, plagiarism, etc.)?",
    "What healthcare services are available to students on campus?",
    "How does the transportation system work for students?",
    "What dining and food services are available at the university?",
    "What are the rules for using IT services and university electronic resources?",
    "What libraries and learning resources are available for students?"
]

In [ ]:
def text_formatter(text:str): 
    text = text.replace("\n", "")
    text = text.replace("  ", "")
    return text

In [ ]:
def open_and_read_file():
    pages_and_chunks  = []
     
    with open("webOrganizedData.txt", "r",  encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines:
            strip_line = line.strip()
            
            metadatas, contexts = strip_line.split("|| dataContent:")
            if len(metadatas)>2: 
                link, name = metadatas.split()
            else:
                name = ""
            
            headers_to_split_on = [("h1", "Header 1")]
            splitter = HTMLSemanticPreservingSplitter(
                headers_to_split_on=headers_to_split_on,
                elements_to_preserve=["table"],
                preserve_links=True,
                max_chunk_size=250
            )
            documents = splitter.split_text(contexts)
            for doc in documents:
                if len(doc.page_content)<=10: 
                    continue
                
                formattedText = text_formatter(doc.page_content).strip()
                chunk_dict = {}
                chunk_dict["line_data"] = "metadata: " + name.replace("\n", "") + " ||content:" + formattedText
                chunk_dict["line_char_count"] = len(formattedText)
                chunk_dict["line_word_count"] = len(doc.page_content.strip().split())
      
                pages_and_chunks.append(chunk_dict)
    return pages_and_chunks

In [ ]:
fetched_texts = open_and_read_file()
demoData = fetched_texts[-40:]

In [ ]:
print(len(fetched_texts))
print(len(demoData))
demoData

In [ ]:
with open("webOrganizedData1111.txt", "a",  encoding="utf-8") as f:
    f.write("\n".join([str(x) for x in fetched_texts]))

<h3>Now work with pdf data</h3>

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


def open_and_read_file_for_pdf_content():
    pages_and_chunks  = []
     
    with open("pdfData.txt", "r",  encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines:
            strip_line = line.strip()
            
            metadatas, contexts = strip_line.split("|| dataContent:")
            
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=250,  
                chunk_overlap=20
            )
            documents = splitter.split_text(contexts) 
            for doc in documents:
                
                formattedText = text_formatter(doc).strip()
                if len(formattedText)<=10: 
                    continue
                chunk_dict = {}
                chunk_dict["line_data"] = "metadata: " + metadatas + " ||content:" + formattedText.replace("#", "")
                chunk_dict["line_char_count"] = len(formattedText)
                chunk_dict["line_word_count"] = len(doc.strip().split())
      
                pages_and_chunks.append(chunk_dict)
    return pages_and_chunks

In [ ]:
arrData = open_and_read_file_for_pdf_content()
print(len(arrData))
print(arrData)

In [ ]:
print(len(fetched_texts))

fetched_texts = fetched_texts + arrData
print(len(fetched_texts))

In [ ]:
dataLines = []
for line in fetched_texts:
    

In [ ]:
df = pd.DataFrame(fetched_texts)
df.describe().round(2)

In [ ]:
converted_line_datas = df.to_dict(orient="records")
converted_line_datas

<h2>Embedding our text chunks</h2>

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="multi-qa-mpnet-base-dot-v1", device="cuda")

# all-mpnet-base-v2
# intfloat/e5-base-v2
# BAAI/bge-base-en
# multi-qa-mpnet-base-dot-v1

In [ ]:
%%time

embedding_model.to("cuda")
for item in tqdm(converted_line_datas):# as "pages_and_chunks_over_min_token_len" contains less informatio and just the links ; I will work with it just to show
    item["embedding"] =embedding_model.encode(item["line_data"])

In [ ]:
line_chunks_and_embeddings_df = pd.DataFrame(converted_line_datas) 
embeddings_df_save_path = "line_chunks_and_embeddings_df.csv"
line_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [ ]:
line_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
line_chunks_and_embedding_df_load.head()

<h2><b>RAG - Search and Answer</b></h2>

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
line_chunks_and_embedding_df = pd.read_csv("line_chunks_and_embeddings_df.csv")

print(line_chunks_and_embedding_df.columns)

In [ ]:
print(line_chunks_and_embedding_df['embedding'].shape)
print(type(line_chunks_and_embedding_df['embedding'][0]))# text_chunks_and_embedding_df['embedding']
line_chunks_and_embedding_df['embedding'][0]

In [ ]:
def convert_to_array(x):
    if isinstance(x, str):  # If stored as string, convert to np.ndarray
        return np.fromstring(x.strip("[]"), sep=" ")
    return x  # If already an array, return as is


In [ ]:
# Convert embedding column back to np.ndarray then tensor
line_chunks_and_embedding_df["embedding"] = line_chunks_and_embedding_df["embedding"].apply(lambda x:convert_to_array(x))
embeddings = torch.tensor(np.stack(line_chunks_and_embedding_df["embedding"].tolist(), axis=0), dtype=torch.float32).to(device)

In [ ]:
print(type(line_chunks_and_embedding_df["embedding"][0])) # it is converted from "str" to "numpy.ndarray"
print(embeddings.shape)
line_chunks_and_embedding_df

In [ ]:
# Convert texts and embedding df to list of dicts
line_and_chunks = line_chunks_and_embedding_df.to_dict(orient="records")
line_and_chunks

<h2>Now, Let's create a small semantic search pipeline.<h2>

In [ ]:
from sentence_transformers import util, SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = SentenceTransformer(model_name_or_path="multi-qa-mpnet-base-dot-v1", device=device)

<b>lets test our retrival part==============================</b>

In [ ]:
import textwrap

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

In [ ]:
queryTest1 = random.choice(gpt4_questions)

queryTest1_embedding = embedding_model.encode(queryTest1, convert_to_tensor=True).to("cuda")

start_time = timer()
dot_scores = util.dot_score(a=queryTest1_embedding, b=embeddings)[0] 
end_time = timer()
print(f"[INFO] Time taken to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")
top_results_dot_product = torch.topk(dot_scores, k=5)
print(top_results_dot_product) 

print(f"\nQuery: '{queryTest1}'\n")
print("Results:")

for probabilityScore, line_index_number in zip(top_results_dot_product[0], top_results_dot_product[1]): # zip(top_results_dot_product.values.cpu().numpy(), top_results_dot_product.indices.cpu().numpy()): 
    print(f"Score: {probabilityScore:.4f}")
    
    print("Output Text:")
    print("Line number: ", line_index_number)
    print_wrapped(line_and_chunks[line_index_number]["line_data"])
    print("\n")

<b>==============================</b>

In [ ]:
def retrieve_relevant_resources(query:str, embeddings:torch.tensor, model:SentenceTransformer=embedding_model, n_resources_to_return: int=5, print_time: bool=True):
    query_embedding = model.encode(query, convert_to_tensor=True)
    
    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] Time taken to get scores on ({len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

    scores, indices = torch.topk(input=dot_scores,k=n_resources_to_return)
    return scores, indices 

In [ ]:
def print_top_results_and_scores(query: str, embeddings: torch.tensor, pages_and_chunks: list[dict]=line_and_chunks, n_resources_to_return: int=5):

    scores, indices = retrieve_relevant_resources(query=query, 
                                                  embeddings=embeddings, 
                                                  n_resources_to_return=n_resources_to_return)

    for score, idx in zip(scores, indices):
        print(f"Score: {score:.4f}")
        print("Text:")
        print_wrapped(line_and_chunks[idx]["line_data"])
        print("\n")

In [ ]:
print_top_results_and_scores(
    query = random.choice(gpt4_questions), 
    embeddings = embeddings
)

<h2><b>Define Groq => llama3-70B model</b></h2> 

In [ ]:
def summerizePrompt(basePrompt:str, temperature:float=1.0, max_completion_tokens:int=1024, top_p:float=1.0, stream:bool=True):
    client = Groq(api_key=api_key)
    completion = client.chat.completions.create(
        model="mistral-saba-24b", # deepseek-r1-distill-llama-70b mistral-saba-24b llama-3.3-70b-versatile
        messages=[
            {
                "role": "system",
                "content":  (
                    "Create answer of user query from the given iser base prompt and ignore the non-related part in the given content"
                    "Rewrite the following passage in a structured human-like manner. "
                    "Remove all asterisks and symbols and replace them with numbers else where appropriate."
                )
            },
            {
                "role": "user",
                "content": basePrompt
            }
        ],
        temperature=1,
        max_completion_tokens=max_completion_tokens,
        top_p=top_p,
        stream=stream,
        stop=None,
    )
    
    
    data = ""
    for chunk in completion:
        data += chunk.choices[0].delta.content or ""
    return data

<h2><b>Augmenting our prompt with context items with prompt engineering</b></h2> 

In [ ]:
def prompt_formatter(query: str, context_items: list[dict]) -> str:
        context = "- " + "\n- ".join([item["line_data"] for item in context_items])
        print(context)
        temp = context.split()
        if len(temp) > 4500:
            temp = temp[:4500]
            
        context = " ".join(temp)
        
        base_prompt = """ 
        Context: {context} 
        User query: {query}
        Answer:
        """ 
        
        base_prompt = base_prompt.format(context=context, query=query)

        return base_prompt

In [ ]:
def ask(query: str, temperature: float=0.3, max_new_tokens:int=256, return_answer_only=True):
    scores, indices = retrieve_relevant_resources(query=query, embeddings=embeddings, n_resources_to_return=10)
    print("Scores: ", scores)
    context_items = [line_and_chunks[i] for i in indices] 
    
    for i, item in enumerate(context_items): 
        item["score"] = scores[i].cpu() 

    # Augment ans
    prompt = prompt_formatter(query=query, context_items=context_items)
    
    #Generating / optimizing ans
    output_text = summerizePrompt(basePrompt=prompt, temperature=temperature, max_completion_tokens=max_new_tokens)
    # output_text = output_text.split("</think>")[1]
    
    if return_answer_only:
        return output_text

    return output_text

In [ ]:
query = random.choice(gpt4_questions)
print(f"Query: {query}")

print(ask(query="all courses needs to taken in 1st year in computer science?",  temperature= 0.6, max_new_tokens=2048, return_answer_only=True))
# max_new_tokens is the max number of tokens to be generated in the response, it is not the max number of tokens in the prompt.

<h2><b>Make API with This Model</b></h2>

In [ ]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

In [ ]:
app = FastAPI()
nest_asyncio.apply()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allow all origins (change this in production)
    allow_credentials=True,
    allow_methods=["*"],  # Allow all HTTP methods
    allow_headers=["*"],  # Allow all headers
)

In [ ]:
class QueryRequest(BaseModel):
    query: str
    isChat: bool

In [ ]:
@app.get("/")
def home():
    return {"answer": "FastAPI is running inside Jupyter Notebook!"}

@app.post("/askQuestion")
@app.post("/askQuestion/")
def askQuestion(data: QueryRequest): 
    query = data.query
    
    answer = ask(query=query, temperature=0.6, max_new_tokens=2048, return_answer_only=True)
    answer = answer.split("\n")
    breakLineAns = "" 
    for ans in answer:
        breakLineAns += ans + "<br>"
    return {"answer": breakLineAns}

In [ ]:
import socket
ip_address = socket.gethostbyname(socket.gethostname())
print(f"Your local IP address: {ip_address}")

In [ ]:
# Run the FastAPI server inside Jupyter
# uvicorn.run(app, host="127.0.0.1", port=8000)

uvicorn.run(app, host="0.0.0.0", port=8709)
# uvicorn.run(app, host="192.168.70.33", port=8709)
# uvicorn.run(app, host="172.29.11.5", port=8709)